# discopipe

> Discord passthrough to a headless coding agent CLI.

One authorized Discord user in one channel talks to `claude -p --continue`
running on this box. Messages go to the agent's stdin; its stdout comes
back as the reply. No prefixes, no commands — pure pass-through.

## Install

```sh
pip install -e .
```

## Configure (environment only)

| var | required | default | meaning |
|---|---|---|---|
| `DISCORD_TOKEN` | yes | — | bot token |
| `DISCOPIPE_USER_ID` | yes | — | the one authorized Discord user (int) |
| `DISCOPIPE_CHANNEL_ID` | yes | — | the one authorized channel (int) |
| `DISCOPIPE_CWD` | yes | — | working directory for the agent — dedicate it to the bot |
| `DISCOPIPE_CMD` | no | `claude -p --continue --dangerously-skip-permissions` | agent command line |
| `DISCOPIPE_TIMEOUT` | no | `600` | wall-clock seconds per run |

**`DISCOPIPE_CWD` must be a directory dedicated to the bot.** `--continue`
resumes "the latest conversation in this directory", so running interactive
`claude` there (e.g. over ssh) silently hijacks the bot's thread.

**Put a `CLAUDE.md` in `DISCOPIPE_CWD`** telling the agent to keep replies
under ~1800 characters (Discord caps messages at 2000; the bot truncates as
a safety net, keeping the tail). Example:

    Replies are read on a phone via Discord. Keep every reply under 1800
    characters. Be terse. For long output, write it to a file and reply
    with the path and a 3-line summary.

**Security note:** the default command includes
`--dangerously-skip-permissions` because headless runs cannot answer
permission prompts. Run this only on a single-purpose box you own; the
Discord side already restricts input to one user and one channel.

## Run

Put the variables in a `chmod 600` env file (e.g.
`~/.config/discopipe/env`) — never on the command line, where the token
would land in shell history. A systemd unit consumes the same file via
`EnvironmentFile=`.

```sh
set -a; source ~/.config/discopipe/env; set +a; discopipe
```

Nonzero agent exits are reported in the reply as `(exit N)` plus the
agent's stderr — never retried, so a transient failure can't replay side
effects (like creating a duplicate PR).

In [ ]:
#| hide
import discopipe